# Public notebook copy

Published from the author's course-project archive. Student identifiers and private paths have been removed; paths now use bundled repository data. Known attribution, numbering, distribution-framing and stale summary errors were corrected. The original private files are unchanged.

Code outputs are cleared so you can run the notebook from top to bottom.

The GPU inference notebook is still missing. These notebooks document the project or analyze precomputed scores. See the repository README for current results and scope.


# F-chelsea-zhao-2: Final Analysis (Notebook N) — ReasonEval on ProcessBench

**MIE1512 Course Project — Final Submission**  
Author: Chelsea Zhao

Reproducible end-to-end analysis on the full 3,400-row ProcessBench dataset using the ReasonEval-Mistral-7B scores that were pre-computed on a local GPU and frozen in `F-data/processbench_full_results.parquet`. This notebook runs on **Colab CPU** — no model loading, no inference. Run Notebook 1 first if you want to regenerate the EDA; this notebook only needs the two Parquet inputs.

- **Parquet inputs only** — no CSV or raw-data downloads at runtime.
- **Runtime → Run All** reproduces every figure and table below.
- All figures and tables use a `full_` filename prefix and land back in `F-data/`.

## Environment and paths

Use Open in Colab, or run from a cloned repository. The bootstrap reads bundled F-data/ inputs and writes outputs to generated-results/. No private Drive mount is needed.


In [ ]:
%pip install -q pandas numpy pyarrow matplotlib seaborn scikit-learn


In [ ]:
# Find the repository data locally, or clone this public repository in Colab.
from pathlib import Path
import os, sys, subprocess

BASE_PATH = Path.cwd()
if not (BASE_PATH / 'F-data').is_dir():
    BASE_PATH = next((p for p in Path.cwd().parents if (p / 'F-data').is_dir()), BASE_PATH)
if not (BASE_PATH / 'F-data').is_dir() and 'google.colab' in sys.modules:
    BASE_PATH = Path('/content/mie1512-reasoneval-portfolio')
    if not BASE_PATH.exists():
        subprocess.run(['git', 'clone', 'https://github.com/rulangyuezhao-cmd/mie1512-reasoneval-portfolio.git', str(BASE_PATH)], check=True)
assert (BASE_PATH / 'F-data').is_dir(), 'Run from a repository checkout, or use Open in Colab.'
DATA_DIR = str(BASE_PATH / 'F-data')
RESULTS_DIR = str(BASE_PATH / 'generated-results')
os.makedirs(RESULTS_DIR, exist_ok=True)
PARQUET_PATH = os.path.join(DATA_DIR, 'processbench_cached.parquet')
SAMPLE_PATH = os.path.join(RESULTS_DIR, 'processbench_sample.parquet')
RESULTS_CSV = os.path.join(DATA_DIR, 'processbench_full_results.parquet')
FIG_PREFIX = 'full_'
print('Bundled inputs:', sorted(os.listdir(DATA_DIR)))
print('Generated files will be saved under generated-results/.')


In [ ]:
import os, time, json, ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from sklearn.metrics import roc_curve
from sklearn.metrics import auc as sklearn_auc
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch
import textwrap

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Generator metadata (family / size / colour) — shared with Notebook 1.
GENERATOR_META = {
    'Meta-Llama-3-8B-Instruct':    {'family': 'Llama-3',      'size': 8,   'color': '#1F4E79'},
    'Meta-Llama-3-70B-Instruct':   {'family': 'Llama-3',      'size': 70,  'color': '#1F4E79'},
    'Llama-3.1-8B-Instruct':       {'family': 'Llama-3.1',    'size': 8,   'color': '#4C72B0'},
    'Llama-3.1-70B-Instruct':      {'family': 'Llama-3.1',    'size': 70,  'color': '#4C72B0'},
    'Qwen2-1.5B-Instruct':         {'family': 'Qwen2',        'size': 1.5, 'color': '#DD8452'},
    'Qwen2-7B-Instruct':           {'family': 'Qwen2',        'size': 7,   'color': '#DD8452'},
    'Qwen2-72B-Instruct':          {'family': 'Qwen2',        'size': 72,  'color': '#DD8452'},
    'Qwen2.5-1.5B-Instruct':       {'family': 'Qwen2.5',      'size': 1.5, 'color': '#55A868'},
    'Qwen2.5-7B-Instruct':         {'family': 'Qwen2.5',      'size': 7,   'color': '#55A868'},
    'Qwen2.5-72B-Instruct':        {'family': 'Qwen2.5',      'size': 72,  'color': '#55A868'},
    'Qwen2.5-Math-7B-Instruct':    {'family': 'Qwen2.5-Math', 'size': 7,   'color': '#C44E52'},
    'Qwen2.5-Math-72B-Instruct':   {'family': 'Qwen2.5-Math', 'size': 72,  'color': '#C44E52'},
}

subsets_order = ['gsm8k', 'math', 'olympiadbench', 'omnimath']
subset_colors = {'gsm8k': '#3498DB', 'math': '#2ECC71',
                 'olympiadbench': '#E74C3C', 'omnimath': '#9B59B6'}

family_models = defaultdict(list)
for gen_name, meta in GENERATOR_META.items():
    family_models[meta['family']].append(gen_name)

PROMPT_FORMAT = (
    'Please help me to evaluate the reasoning steps of a given question and determine '
    'if they contain any errors.\nQuestion: {input}\nSolution: '
)
print('Imports and metadata ready.')

## Load input data

Read benchmark labels and the precomputed full scoring artifact from the bundled Parquet files. The local GPU inference notebook is not included.


In [ ]:
# Full ProcessBench — used for label alignment throughout analysis
df_pb = pd.read_parquet(PARQUET_PATH)
print(f'Full ProcessBench: {len(df_pb):,} rows')

# Full-dataset ReasonEval results (from F-chelsea-zhao-2)
df_results = pd.read_parquet(RESULTS_CSV)
print(f'Full inference results: {len(df_results):,} rows')
print(df_results.groupby('subset').size().to_dict())

if len(df_results) < len(df_pb):
    print(f'\nWARNING: {len(df_pb) - len(df_results):,} rows missing from results — '
          f'F-chelsea-zhao-2 may not have finished. Analysis will proceed on {len(df_results):,} rows.')

---

## 6. Figure 1 — Qualitative Step Scoring

### Paper Reference: Section 1, Figure 1 (p.2)

Reproduces Figure 1: per-step S_validity and S_redundancy scores visualised as colour-coded cards for a representative solution, with the ground-truth error step highlighted in red. Demonstrates ReasonEval's ability to localise errors at the step level.

> **GenAI prompt used:** *"Create a matplotlib card-style figure showing per-step S_validity and S_redundancy scores for a selected solution, with the ground-truth error step highlighted in red, other invalid steps in orange, and valid steps in grey. Include problem text, step text, scores, and flags in each card."*

In [ ]:
def parse_score_list(x):
    if x is None: return None
    if isinstance(x, str): x = ast.literal_eval(x)
    if isinstance(x, (list, tuple, np.ndarray)): return [float(v) for v in x]
    if pd.isna(x): return None
    raise TypeError('Unsupported score-list value')


def sanitize_for_mpl(text: str) -> str:
    """Escape LaTeX commands so matplotlib mathtext parser doesn't choke."""
    text = str(text)
    text = text.replace('\\', r'\\')   # must be first
    text = text.replace('$', r'\$')
    return text

def wrap_text(text, width=95):
    return "\n".join(textwrap.wrap(sanitize_for_mpl(str(text)), width=width))

df_scored = df_results.copy()
for col in ['step_validity', 'step_redundancy']:
    df_scored[col] = df_scored[col].apply(parse_score_list)

candidates = df_scored[
    (df_scored['has_error'] == 1) &
    (df_scored['solution_validity'] < 0.5) &
    (df_scored['subset'].isin(['math', 'gsm8k'])) &
    (df_scored['step_validity'].notna())
].head(10)

if len(candidates) == 0:
    print('No matching candidates found — check inference results.')
else:
    ex_id = candidates.iloc[0]['id']
    ex_pb = df_pb[df_pb['id'] == ex_id].iloc[0]
    ex_sc = df_scored[df_scored['id'] == ex_id].iloc[0]

    steps    = ex_pb['steps']
    gt_label = int(ex_pb['label'])
    sv_list  = [float(v) for v in ex_sc['step_validity']]
    sr_list  = [float(v) for v in ex_sc['step_redundancy']]

    n_rows = min(len(steps), len(sv_list), len(sr_list))
    steps, sv_list, sr_list = steps[:n_rows], sv_list[:n_rows], sr_list[:n_rows]

    wrapped_steps = [wrap_text(step, width=95) for step in steps]
    line_counts   = [max(2, s.count('\n') + 1) for s in wrapped_steps]
    block_heights = [0.7 + 0.35 * lc for lc in line_counts]
    total_height  = sum(block_heights) + 2.8

    fig, ax = plt.subplots(figsize=(16, total_height))
    ax.set_xlim(0, 1); ax.set_ylim(0, total_height); ax.axis('off')

    # Title — sanitize problem text independently (wider wrap)
    problem_wrapped = wrap_text(ex_pb['problem'], width=120)
    title = (f"Problem: {problem_wrapped}\n"
             f"Subset: {ex_pb['subset'].upper()}   |   ID: {ex_id}   |   "
             f"GT first error: Step {gt_label} (0-indexed)")
    ax.text(0.02, total_height - 0.35, title,
            fontsize=11, va='top', ha='left', weight='bold')

    y = total_height - 2.0
    for i, (step_text, sv, sr, h) in enumerate(zip(wrapped_steps, sv_list, sr_list, block_heights)):
        face = '#FDE2E1' if i == gt_label else ('#FFF0D9' if sv < 0.5 else '#F8F9FA')
        edge = '#D9534F' if i == gt_label else ('#F0AD4E' if sv < 0.5 else '#C7CDD4')

        flags = []
        if sv < 0.25:   flags.append('INVALID < 0.25')
        elif sv < 0.5:  flags.append('INVALID < 0.5')
        if sr > 0.15:   flags.append('REDUNDANT')
        flag_text = ' | '.join(flags) if flags else 'valid'
        gt_text   = '  |  GT ERROR' if i == gt_label else ''

        box = FancyBboxPatch((0.02, y - h), 0.96, h,
                             boxstyle="round,pad=0.012,rounding_size=0.015",
                             linewidth=1.6, edgecolor=edge, facecolor=face)
        ax.add_patch(box)
        ax.text(0.04, y - 0.28,
                f"Step {i}{gt_text}",
                fontsize=10.5, weight='bold', va='top')
        ax.text(0.04, y - 0.68,
                f"S_validity = {sv:.4f}    |    S_redundancy = {sr:.4f}    |    Flag = {flag_text}",
                fontsize=9.5, va='top')
        ax.text(0.04, y - 1.12, step_text, fontsize=10, va='top')
        y -= h + 0.18

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'full_figure1_step_example_cards.png'),
                bbox_inches='tight', dpi=160)
    plt.show()
    print(f'Figure 1 saved. Selected example ID: {ex_id}')

---

## 7. Metric Helper Functions

### Paper Reference: Sections 4.2 and 5.1 (p.4–6)

Ported from the official ReasonEval evaluation scripts (`get_mr_math_results.py`,
`get_mr_gsm8k_results.py`) [7].

**Critical convention (Paper §4.2, p.5):**
ProcessBench `label` is 0-indexed; the paper uses 1-indexed step counts.
`compute_metrics()` applies `k = label + 1` for step-level alignment.

**Thresholds (Appendix D/E, p.12):**
- `threshold=0.5` for F1 Score (Tables 1 & 2)
- `threshold=0.25` for False Positive Rate (Table 4)


### 7.1 Define Metric Functions

> **GenAI prompt used:** *"Port macro_f1() and compute_metrics() from the official ReasonEval GitHub evaluation scripts. compute_metrics() should accept df_pb_sub (with steps and label columns) and df_sc_sub (with solution_validity, step_validity columns), merge on id, compute macro F1 and ROC-AUC at both solution and step level, and return a dict with sol_f1, sol_auc, step_f1, step_auc as percentages."*

In [ ]:
import ast
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc as sklearn_auc


def macro_f1(ct, tt, tf, cf):
    """Macro-averaged F1 over correct (positive) and incorrect (negative) classes.
    
    Args:
        ct: correctly classified true (correct solutions predicted correct)
        tt: total true (correct solutions)
        tf: total false (incorrect solutions)
        cf: correctly classified false (incorrect solutions predicted incorrect)
    """
    if tt == 0 or tf == 0:
        return float('nan')
    tp, fp = ct, tf - cf
    fn, tn = tt - ct, cf

    pp = tp / (tp + fp) if (tp + fp) else 0
    rp = tp / (tp + fn) if (tp + fn) else 0
    f1p = 2 * pp * rp / (pp + rp) if (pp + rp) else 0

    pn = tn / (tn + fn) if (tn + fn) else 0
    rn = tn / (tn + fp) if (tn + fp) else 0
    f1n = 2 * pn * rn / (pn + rn) if (pn + rn) else 0

    return (f1p + f1n) / 2


def parse_score_list(x):
    if x is None: return None
    if isinstance(x, str): x = ast.literal_eval(x)
    if isinstance(x, (list, tuple, np.ndarray)): return [float(v) for v in x]
    if pd.isna(x): return None
    raise TypeError('Unsupported score-list value')


def compute_metrics(df_pb_sub, df_sc_sub, threshold=0.5):
    """Compute solution-level and step-level F1 and AUC against ProcessBench labels.

    ProcessBench convention:
      label == -1  → correct solution (no error)
      label >= 0   → 0-indexed position of first erroneous step

    ReasonEval paper convention (Appendix D/E):
      validity < threshold  → step/solution flagged as invalid
      k = label + 1         → 1-indexed first-error step count

    Step-level ground truth:
      Correct solution  : all steps labelled 1 (valid)
      Incorrect solution: steps 0..(k-2) labelled 1, step k-1 labelled 0,
                          steps k.. labelled 'N/A' (excluded from metrics)

    Returns dict with sol_f1, sol_auc, step_f1, step_auc as percentages (0–100).
    """
    # Counters for macro-F1
    sol_ct = sol_cf = sol_tt = sol_tf = 0
    stp_ct = stp_cf = stp_tt = stp_tf = 0

    # Score/label lists for AUC
    sol_pred, sol_tgt = [], []
    stp_pred, stp_tgt = [], []

    # Parse score columns
    df_sc_sub = df_sc_sub.copy()
    df_sc_sub['solution_validity']   = pd.to_numeric(df_sc_sub['solution_validity'],   errors='coerce')
    df_sc_sub['solution_redundancy'] = pd.to_numeric(df_sc_sub['solution_redundancy'], errors='coerce')
    df_sc_sub['step_validity']       = df_sc_sub['step_validity'].apply(parse_score_list)
    df_sc_sub['step_redundancy']     = df_sc_sub['step_redundancy'].apply(parse_score_list)

    df_merged = pd.merge(df_pb_sub, df_sc_sub, on='id', how='inner', suffixes=('_pb', '_sc'))

    for _, row in df_merged.iterrows():
        label   = int(row['label_pb'])
        sol_sv  = float(row['solution_validity'])
        sv_list = row['step_validity']

        if sv_list is None or pd.isna(sol_sv):
            continue

        sv_list = [float(v) for v in sv_list]
        n       = min(len(row['steps']), len(sv_list))
        sv_list = sv_list[:n]
        correct = (label == -1)

        # ── Solution-level ────────────────────────────────────────────────────
        if correct:
            sol_tt += 1
            if sol_sv > threshold:
                sol_ct += 1          # correctly predicted valid
            sol_tgt.append(1)
            sol_pred.append(sol_sv)
        else:
            sol_tf += 1
            if sol_sv <= threshold:
                sol_cf += 1          # correctly predicted invalid
            sol_tgt.append(0)
            sol_pred.append(sol_sv)

        # ── Step-level ────────────────────────────────────────────────────────
        # Build ground-truth step labels
        if correct:
            gt_s = [1] * n
        else:
            k = label + 1           # 0-indexed → 1-indexed first-error position
            if k > n:
                continue            # label out of range for this solution
            gt_s = [1] * (k - 1) + [0] + ['N/A'] * (n - k)

        pr_s = [1 if sv > threshold else 0 for sv in sv_list]

        for g, p in zip(gt_s, pr_s):
            if g == 1:
                stp_tt += 1
                if p == 1:
                    stp_ct += 1
            elif g == 0:
                stp_tf += 1
                if p == 0:
                    stp_cf += 1
            # g == 'N/A' → skip (post-error steps excluded per paper)

        # AUC: include only steps up to and including the first error
        if correct:
            stp_tgt.extend([1] * n)
            stp_pred.extend(sv_list)
        else:
            k = label + 1
            stp_tgt.extend([1] * (k - 1) + [0])
            stp_pred.extend(sv_list[:k])

    # ── Aggregate metrics ─────────────────────────────────────────────────────
    sol_f1 = macro_f1(sol_ct, sol_tt, sol_tf, sol_cf)
    stp_f1 = macro_f1(stp_ct, stp_tt, stp_tf, stp_cf)

    try:
        fpr, tpr, _ = roc_curve(sol_tgt, sol_pred)
        sol_auc = sklearn_auc(fpr, tpr)
    except Exception:
        sol_auc = float('nan')

    try:
        fpr, tpr, _ = roc_curve(stp_tgt, stp_pred)
        stp_auc = sklearn_auc(fpr, tpr)
    except Exception:
        stp_auc = float('nan')

    return {
        'sol_f1':   round(sol_f1  * 100, 1),
        'sol_auc':  round(sol_auc * 100, 1),
        'step_f1':  round(stp_f1  * 100, 1),
        'step_auc': round(stp_auc * 100, 1),
    }


print('Metric functions defined: macro_f1(), parse_score_list(), compute_metrics()')

## Detection metrics and distribution framing

Compute solution- and step-level F1/AUC on the bundled ProcessBench data. MATH is a same-source-domain reference relative to PRM800K; GSM8K is an OOD grade-school domain, and OlympiadBench/OmniMath are harder OOD domains. These are domain-based descriptions, not exact distribution-match claims.

The hardcoded MR-MATH and MR-GSM8K table values are paper reference values, not results reproduced on the original evaluation datasets. Different generators and annotation settings prevent direct benchmark comparisons.


### 8.1 Compute and Display Metrics

> **GenAI prompt used:** *"Call compute_metrics() for each ProcessBench subset and an overall row, display results alongside paper Table 1 and Table 2 snippets, and label each as approximate in-distribution or out-of-distribution comparisons."*

In [ ]:
# ── Paper reference values ────────────────────────────────────────────────
df_paper_table1 = pd.DataFrame([{
    'Evaluator': 'ReasonEval-Mistral-7B', 'dataset': 'MR-MATH-invalid (paper Table 1)',
    'Sol F1 (%)': 78.0, 'Sol AUC (%)': 85.1, 'Step F1 (%)': 68.6, 'Step AUC (%)': 85.7,
}])

df_paper_table2 = pd.DataFrame([{
    'Evaluator': 'ReasonEval-Mistral-7B', 'dataset': 'MR-GSM8K-original (paper Table 2)',
    'Sol F1 (%)': 61.8, 'Sol AUC (%)': 79.8, 'Step F1 (%)': 62.9, 'Step AUC (%)': 86.1,
}])

# ── ProcessBench results ─────────────────────────────────────────────────
rows = []
for s in subsets_order:
    m = compute_metrics(df_pb[df_pb['subset'] == s], df_results[df_results['subset'] == s])
    rows.append({'Subset': s.upper(), **m})

m_all = compute_metrics(df_pb, df_results)
rows.append({'Subset': 'OVERALL', **m_all})

df_t1 = pd.DataFrame(rows).rename(columns={
    'sol_f1': 'Sol F1 (%)', 'sol_auc': 'Sol AUC (%)',
    'step_f1': 'Step F1 (%)', 'step_auc': 'Step AUC (%)'
})
df_t1['Evaluator'] = 'ReasonEval-Mistral-7B'
df_t1['OOD note'] = df_t1['Subset'].map({
    'GSM8K': 'OOD: grade-school domain',
    'MATH': 'MATH-domain reference (ID-like)',
    'OLYMPIADBENCH': 'out-of-dist (harder than training)',
    'OMNIMATH': 'out-of-dist (hardest)',
    'OVERALL': '—'
})

print('Paper Table 1 (MR-MATH-invalid):')
print(df_paper_table1.to_string(index=False))

print('\nPaper Table 2 (MR-GSM8K-original):')
print(df_paper_table2.to_string(index=False))

print('\nThis project — ProcessBench subsets:')
print(df_t1[['Subset', 'Sol F1 (%)', 'Sol AUC (%)', 'Step F1 (%)', 'Step AUC (%)', 'OOD note']].to_string(index=False))

---

## 9. Figure 2 — Generator Accuracy vs AUC

### Paper Reference: Section 4.2, Figure 2 (p.5)

Reproduces Figure 2: X-axis = generator final-answer accuracy (Pass@1 proxy);
Y-axis = ReasonEval solution-level AUC; bubble size = model parameter count;
colour = generator family.

**Expected finding (paper):** Generators with higher final-answer accuracy tend to produce
solutions where ReasonEval achieves higher AUC.


### 9.1 Compute Per-Generator Accuracy and AUC

> **GenAI prompt used:** *"Merge df_results with df_pb on id, iterate over generators computing final-answer accuracy and compute_metrics() AUC, and store results in df_gen with family, size, and color metadata for plotting."*

In [ ]:
df_joined = df_results.merge(df_pb, on='id', how='inner', suffixes=('_sc', '_pb'))

df_joined = df_results.merge(df_pb, on='id', how='inner', suffixes=('_sc', '_pb'))

# Resolve 'generator' regardless of which side it came from / whether it got suffixed
if 'generator' not in df_joined.columns:
    if 'generator_sc' in df_joined.columns:
        df_joined['generator'] = df_joined['generator_sc']
    elif 'generator_pb' in df_joined.columns:
        df_joined['generator'] = df_joined['generator_pb']
        
gen_rows = []
for gen in df_joined['generator'].dropna().unique():
    g = df_joined[df_joined['generator'] == gen].copy()
    if len(g) < 3: continue

    pb_g = g[['id', 'label_pb', 'has_error_pb', 'final_answer_correct_pb', 'n_steps_pb', 'steps']].copy()
    sc_g = g[['id', 'solution_validity', 'solution_redundancy', 'step_validity', 'step_redundancy']].copy()

    acc     = g['final_answer_correct_pb'].mean() * 100
    metrics = compute_metrics(pb_g, sc_g)
    meta    = GENERATOR_META.get(gen, {'family': 'Unknown', 'size': 7, 'color': 'grey'})

    gen_rows.append({
        'generator': gen, 'family': meta['family'],
        'size': meta['size'], 'color': meta['color'],
        'acc': acc, 'sol_auc': metrics['sol_auc'],
    })

df_gen = pd.DataFrame(gen_rows)
print(df_gen[['generator', 'acc', 'sol_auc']].to_string(index=False))

### 9.2 Plot Figure 2

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sz_map = {1.5: 70, 7: 120, 8: 120, 70: 200, 72: 200}

for _, row in df_gen.iterrows():
    s = sz_map.get(row['size'], 100)
    ax.scatter(row['acc'], row['sol_auc'], color=row['color'], s=s,
               edgecolors='black', linewidths=0.7, zorder=4)
    offset = (10, 8) if row['acc'] < 55 else (-60, 5)
    ax.annotate(row['generator'].replace('-Instruct', ''), (row['acc'], row['sol_auc']),
                textcoords='offset points', xytext=offset, fontsize=8)

fam_colors = {m['family']: m['color'] for m in GENERATOR_META.values()}
leg_fam  = [mpatches.Patch(color=c, label=f) for f, c in sorted(fam_colors.items())]
leg_size = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', markersize=6,  label='1.5B'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', markersize=9,  label='7B/8B'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', markersize=13, label='70B/72B'),
]
ax.legend(handles=leg_fam + leg_size, fontsize=8, loc='upper left',
          bbox_to_anchor=(1.02, 1), borderaxespad=0, frameon=False, ncol=2)
ax.set_xlabel('Final-answer Accuracy / Pass@1 Proxy (%)', fontsize=11)
ax.set_ylabel('Solution-level AUC (%)', fontsize=11)
ax.set_title('Figure 2 — Generator Accuracy vs ReasonEval Detection AUC\n'
             '(ProcessBench, ReasonEval-Mistral-7B)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'full_figure2_acc_vs_auc.png'), bbox_inches='tight', dpi=130)
plt.show()
print('Figure 2 saved.')

---

## 10. Figure 3 — Score Distributions

### Paper Reference: Section 5.1, Figure 3 (p.6)

Reproduces Figure 3: dual box plot showing S_validity (upper) and S_redundancy (lower)
distributions by generator, split by final-answer correctness (correct vs incorrect).
Generators grouped by family (Llama left, Qwen right).

**Expected finding (paper):** S_validity scores should be clearly lower for solutions with
incorrect final answers, indicating ReasonEval's discriminative ability.


### 10.1 Plot Figure 3

> **GenAI prompt used:** *"Create a two-panel seaborn boxplot figure showing solution_validity (top) and solution_redundancy (bottom) distributions by generator, split by final_answer_correct, with generators ordered by family (Llama then Qwen), family background shading, threshold lines at 0.5/0.25/0.15, and a shared legend."*

In [ ]:
df_plot = df_joined[df_joined['solution_validity'].notna()].copy()
df_plot['answer_label'] = df_plot['final_answer_correct_pb'].map(
    {True: 'Correct final answer', False: 'Incorrect final answer'})
df_plot['generator_short'] = df_plot['generator'].str.replace('-Instruct', '', regex=False)

meta_df = df_plot[['generator', 'generator_short']].drop_duplicates().copy()
meta_df['family'] = meta_df['generator'].map({k: v['family'] for k, v in GENERATOR_META.items()}).fillna('Unknown')
meta_df['size']   = meta_df['generator'].map({k: v['size']   for k, v in GENERATOR_META.items()}).fillna(0)
family_rank = {'Llama-3': 0, 'Llama-3.1': 1, 'Qwen2': 2, 'Qwen2.5': 3, 'Qwen2.5-Math': 4, 'Unknown': 99}
meta_df['family_rank'] = meta_df['family'].map(family_rank).fillna(99)
meta_df = meta_df.sort_values(['family_rank', 'size', 'generator_short']).reset_index(drop=True)
gen_order = meta_df['generator_short'].tolist()
qwen_start_idx = next((i for i, g in enumerate(gen_order) if 'qwen' in g.lower()), None)

pal = {'Correct final answer': '#4C72B0', 'Incorrect final answer': '#DD8452'}
fig, axes = plt.subplots(2, 1, figsize=(18.5, 10.2), sharex=True, gridspec_kw={'hspace': 0.16})

common_box_kws = dict(hue='answer_label', palette=pal, order=gen_order, width=0.8,
    linewidth=1.15, dodge=True,
    flierprops=dict(marker='o', markersize=2.2, markerfacecolor='#888888', markeredgecolor='none', alpha=0.30),
    whiskerprops=dict(linewidth=1.1, color='#666666'), capprops=dict(linewidth=1.1, color='#666666'),
    medianprops=dict(linewidth=2.0, color='#2F2F2F'), boxprops=dict(edgecolor='#555555', linewidth=1.1))

for ax in axes:
    if qwen_start_idx is not None:
        ax.axvspan(-0.5, qwen_start_idx - 0.5, color='#4C72B0', alpha=0.045, zorder=0)
        ax.axvspan(qwen_start_idx - 0.5, len(gen_order) - 0.5, color='#DD8452', alpha=0.040, zorder=0)
        ax.axvline(qwen_start_idx - 0.5, color='#8A8A8A', linestyle='--', lw=1.1, alpha=0.9, zorder=1)

sns.boxplot(data=df_plot, x='generator_short', y='solution_validity', ax=axes[0], **common_box_kws)
axes[0].axhline(0.5, color='#2E8B57', linestyle='--', lw=1.35, alpha=0.9)
axes[0].axhline(0.25, color='#55A868', linestyle=':', lw=1.35, alpha=0.95)
axes[0].set_ylabel(r'$S_{\mathrm{validity}}$', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylim(0, 1.0)
axes[0].set_title('Figure 3 — Distribution of $S_{validity}$ and $S_{redundancy}$ by Generator\n'
                  'ProcessBench evaluated with ReasonEval-Mistral-7B', fontsize=14, fontweight='bold', pad=16)

sns.boxplot(data=df_plot, x='generator_short', y='solution_redundancy', ax=axes[1], **common_box_kws)
axes[1].axhline(0.15, color='#7FB77E', linestyle='--', lw=1.35, alpha=0.95)
axes[1].set_ylabel(r'$S_{\mathrm{redundancy}}$', fontsize=13)
axes[1].set_xlabel('Generator', fontsize=12)
axes[1].set_ylim(0, 0.75)

for ax in axes:
    leg = ax.get_legend()
    if leg: leg.remove()
    ax.grid(axis='y', color='#D9D9D9', linewidth=0.85, alpha=0.85)
    ax.grid(axis='x', visible=False)
    ax.spines[['top','right']].set_visible(False)

axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=32, ha='right')

if qwen_start_idx is not None:
    llama_center = (qwen_start_idx - 1) / 2
    qwen_center  = (qwen_start_idx + len(gen_order) - 1) / 2
    for ax, y_pos in zip(axes, [1.02, 1.04]):
        ax.text(llama_center, y_pos, 'Llama family', transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2F2F2F')
        ax.text(qwen_center, y_pos, 'Qwen family', transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2F2F2F')

legend_handles = [
    Patch(facecolor=pal['Incorrect final answer'], edgecolor='#555555', label='Incorrect final answer'),
    Patch(facecolor=pal['Correct final answer'],   edgecolor='#555555', label='Correct final answer'),
    Line2D([0],[0], color='#2E8B57', linestyle='--', lw=1.35, label='Threshold 0.50 (F1)'),
    Line2D([0],[0], color='#55A868', linestyle=':',  lw=1.35, label='Threshold 0.25 (FPR)'),
    Line2D([0],[0], color='#7FB77E', linestyle='--', lw=1.35, label='Threshold 0.15'),
]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.995),
           ncol=5, frameon=False, fontsize=10, handlelength=2.2, columnspacing=1.4)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(RESULTS_DIR, 'full_figure3_score_distributions.png'), bbox_inches='tight', dpi=220)
plt.show()
print('Figure 3 saved.')

## Low-validity flags among correct final answers

Historical FPR terminology is retained in the tables for traceability. The computed quantity is the fraction of correct-final-answer solutions with validity below 0.25; a correct answer can still have a reasoning error.


### 11.1 Compute FPR per Generator

> **GenAI prompt used:** *"Compute False Positive Rate per generator from df_joined: for correct-final-answer solutions, FPR = proportion where solution_validity < 0.25. Return a DataFrame sorted by accuracy with generator, family, size, accuracy, N correct, N flagged, FPR columns."*

In [ ]:
FPR_THRESH = 0.25
t4_rows = []

df_r = df_joined[df_joined['solution_validity'].notna()].copy()

for gen in sorted(df_r['generator'].dropna().unique()):
    g = df_r[df_r['generator'] == gen].copy()
    acc = g['final_answer_correct_pb'].mean() * 100
    correct_sc = g[g['final_answer_correct_pb'] == True]

    if len(correct_sc) == 0:
        fpr, n_flagged = float('nan'), 0
    else:
        flagged   = (correct_sc['solution_validity'] < FPR_THRESH).sum()
        fpr       = flagged / len(correct_sc) * 100
        n_flagged = int(flagged)

    meta = GENERATOR_META.get(gen, {'family': 'Unknown', 'size': '?'})
    t4_rows.append({
        'Generator': gen, 'Family': meta['family'], 'Size (B)': meta['size'],
        'Acc. (%)': round(acc, 1), 'N correct': len(correct_sc),
        'N flagged': n_flagged,
        'FPR (%)': round(fpr, 1) if pd.notna(fpr) else np.nan,
    })

df_t4 = pd.DataFrame(t4_rows).sort_values('Acc. (%)')
print(f'Table 4 — FPR per Generator  (S_validity threshold = {FPR_THRESH})')
print(df_t4.to_string(index=False))
df_t4.to_csv(os.path.join(RESULTS_DIR, 'full_table4_fpr.csv'), index=False)
print('Table 4 saved.')

---

## 12. Figure 4 — Accuracy vs FPR Scatter

### Paper Reference: Section 5.1, Figure 4 (p.7)

Reproduces Figure 4: original paper models (Abel/WizardMath/LLaMA-PRM800K, hardcoded from
Table 4) overlaid with ProcessBench generators, plotting final-answer accuracy (x-axis)
vs False Positive Rate (y-axis).

**Expected finding (paper):** Negative correlation — higher accuracy models achieve lower FPR.
This project tests whether the same trend holds for Llama/Qwen generators.


### 12.1 Plot Figure 4

> **GenAI prompt used:** *"Create a scatter plot overlaying original paper models (hardcoded from Table 4) as grey markers and ProcessBench generators as coloured markers by family, with marker shapes indicating model size (circle=7B, square=13B, triangle=70B), x-axis=accuracy, y-axis=FPR, and a combined legend."*

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

# Original paper models (Table 4 values, hardcoded)
paper_models = [
    ('LLaMA13B-PRM800K',   0.074, 0.406, 'square',   '#888888', True),
    ('LLaMA70B-PRM800K',   0.147, 0.218, 'triangle', '#888888', True),
    ('Abel7B-001',         0.124, 0.318, 'circle',   '#888888', True),
    ('Abel13B',            0.152, 0.298, 'square',   '#888888', True),
    ('Abel70B',            0.257, 0.200, 'triangle', '#888888', True),
    ('Abel7B-002',         0.300, 0.228, 'circle',   '#888888', False),
    ('WizardMath7B-V1.0',  0.089, 0.344, 'circle',   '#aaaaaa', True),
    ('WizardMath13B',      0.109, 0.313, 'square',   '#aaaaaa', True),
    ('WizardMath70B-V1.0', 0.187, 0.165, 'triangle', '#aaaaaa', True),
    ('WizardMath7B-V1.1',  0.310, 0.167, 'circle',   '#aaaaaa', False),
]

label_map = {
    'Qwen2.5-1.5B-Instruct': 'Qwen2.5-1.5B', 'Llama-3.1-8B-Instruct': 'Llama-3.1-8B',
    'Qwen2-72B-Instruct': 'Qwen2-72B', 'Llama-3.1-70B-Instruct': 'Llama-3.1-70B',
    'Qwen2.5-Math-7B-Instruct': 'Qwen2.5-Math-7B', 'Qwen2.5-Math-72B-Instruct': 'Qwen2.5-Math-72B',
    'Meta-Llama-3-70B-Instruct': 'Meta-Llama-3-70B', 'Meta-Llama-3-8B-Instruct': 'Meta-Llama-3-8B',
    'Qwen2-7B-Instruct': 'Qwen2-7B', 'Qwen2.5-72B-Instruct': 'Qwen2.5-72B',
    'Qwen2-1.5B-Instruct': 'Qwen2-1.5B', 'Qwen2.5-7B-Instruct': 'Qwen2.5-7B',
}

mm = {'circle': 'o', 'square': 's', 'triangle': '^'}
sm = {'circle': 100, 'square': 100, 'triangle': 130}

for name, acc, fpr, shape, color, filled in paper_models:
    fc = color if filled else 'white'
    ax.scatter(acc, fpr, marker=mm[shape], s=sm[shape], facecolors=fc, edgecolors=color,
               linewidths=1.5, zorder=3, alpha=0.8)
    ax.annotate(name, (acc, fpr), textcoords='offset points', xytext=(7, 3),
                fontsize=7, color='#555555')

def sz_marker(size_b):
    if size_b >= 70: return '^'
    elif size_b >= 10: return 's'
    else: return 'o'

for _, row in df_t4.iterrows():
    if pd.isna(row['FPR (%)']): continue
    meta = GENERATOR_META.get(row['Generator'], {'family': 'Unknown', 'size': 7, 'color': 'grey'})
    x = row['Acc. (%)'] / 100
    y = row['FPR (%)'] / 100
    if y == 0:  # jitter zero-FPR points to avoid overlap
        x += np.random.uniform(-0.01, 0.01)
        y += np.random.uniform(0.0005, 0.01)
    ax.scatter(x, y, marker=sz_marker(meta['size']), s=170, facecolors=meta['color'],
               edgecolors='black', linewidths=1.2, zorder=5)
    ax.annotate(label_map.get(row['Generator'], row['Generator'].replace('-Instruct', '')),
                (x, y), textcoords='offset points', xytext=(10, 8), fontsize=7.5,
                color='black', fontweight='bold')

fam_colors  = {m['family']: m['color'] for m in GENERATOR_META.values()}
pb_patches  = [mpatches.Patch(color=c, label=f'PB: {f}') for f, c in sorted(fam_colors.items())]
paper_items = [
    mpatches.Patch(color='#888888', label='Paper: Abel/LLaMA'),
    mpatches.Patch(color='#aaaaaa', label='Paper: WizardMath'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', markersize=7, label='7B/8B'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='grey', markersize=7, label='13B'),
    Line2D([0],[0], marker='^', color='w', markerfacecolor='grey', markersize=9, label='70B/72B'),
]
ax.legend(handles=pb_patches + paper_items, fontsize=7.5, loc='upper right', ncol=2)
ax.set_xlabel('Final-answer Accuracy', fontsize=11)
ax.set_ylabel('False Positive Rate', fontsize=11)
ax.set_title('Figure 4 — Final-answer Accuracy vs False Positive Rate\n'
             'Original paper models (grey) + ProcessBench generators (coloured by family)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'full_figure4_acc_vs_fpr.png'), bbox_inches='tight', dpi=130)
plt.show()
print('Figure 4 saved.')

## Summary and limitations (updated for the full saved run)

The bundled score artifact covers 3,400 unique solutions. Inspect the computed tables above for the actual metrics; earlier sample-era narrative about below-chance OlympiadBench AUC was stale and has been removed.

- This notebook analyzes existing ReasonEval-7B scores; it does not load an LLM or reproduce GPU inference.
- ProcessBench provides first-error labels, not human redundancy annotations.
- MATH is an ID-like domain reference; GSM8K, OlympiadBench and OmniMath test different domains. Generator styles also differ from training.
- The historical FPR field measures low-validity flags among solutions with correct final answers. It is not a false-positive rate against human reasoning-validity labels.
- Paper-model overlays are context from different evaluation settings, not a controlled comparison.
- Future calibration work follows Park et al., Know What You Don't Know (NeurIPS 2025), https://arxiv.org/abs/2506.09338; it is not a completed calibrated result.


---

## 14. References

1. Xia, S., Li, X., Liu, Y., Wu, T., and Liu, P.
   "Evaluating Mathematical Reasoning Beyond Accuracy." *Proceedings of the AAAI Conference on
   Artificial Intelligence (AAAI 2025).* arXiv:2404.05692.

2. Zheng, C., et al. "ProcessBench: Identifying Process Errors in Mathematical Reasoning."
   *Annual Meeting of the Association for Computational Linguistics (ACL 2025).* arXiv:2412.06559.

3. Wang, P., et al. "Math-Shepherd: Verify and Reinforce LLMs Step-by-step without Human Annotations."
   In *Proc. 62nd ACL*, Bangkok, Thailand, 2024, pp. 9426–9439. DOI: 10.18653/v1/2024.acl-long.510

4. Numina. "NuminaMath-CoT." HuggingFace, 2024.
   Available: https://huggingface.co/datasets/AI-MO/NuminaMath-CoT

5. Toshniwal, S., et al. "OpenMathInstruct-1: A 1.8 Million Math Instruction Tuning Dataset."
   arXiv:2402.10176, 2024.

6. Toshniwal, S., et al. "OpenMathInstruct-2: Accelerating AI for Math with Massive Open-Source
   Instruction Data." arXiv:2410.01560, 2024.

7. GAIR-NLP. "ReasonEval." GitHub, 2024.
   Available: https://github.com/GAIR-NLP/ReasonEval
